# Stage-21 inverse study + bias diagnostic (Google Colab)

This notebook runs the **verified** Stage-21 reruns with the final code:

1. `scripts/run_stage21_colab.py` — 2,700-realization blind inverse study (15 states × 3 protocols × 3 noise levels × 20 realizations), with per-realization standard errors, 95% intervals, convergence/stopping flags and timings.
2. `scripts/run_stage21_bias_real.py` — the 3-way bias diagnostic (D1 same-grid noise-free, D2 cross-grid noise-free, D3 warm-vs-cold at 1% noise).
3. `scripts/summarize_stage21_real.py` — recomputes every summary from the individual records.

The run is **resumable**: progress is written every 50 realizations to `results/checkpoints/inverse_records.pkl`. If Colab disconnects, re-run the same cells and it continues from the checkpoint.

**Runtime notes**

- Measured cost is ~30 s CPU per realization, i.e. ~22 CPU-hours total.
- Free Colab (2 vCPU): ~11–13 h. Colab Pro / high-RAM (4–8 vCPU): ~3–6 h.
- Keep the project folder inside Google Drive so checkpoints survive disconnects.
- Before starting, let Drive finish syncing `results/checkpoints/` from the Mac (the local run already saved 100 verified realizations).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, glob, subprocess

# Set this if auto-detection fails, e.g.
# PROJECT_DIR = '/content/drive/MyDrive/PINN to a Neural Operator'
PROJECT_DIR = globals().get('PROJECT_DIR', '')

candidates = [
    PROJECT_DIR,
    '/content/drive/MyDrive/PINN to a Neural Operator',
    '/content/drive/MyDrive/Google drive MAC/PINN to a Neural Operator',
]
for c in candidates:
    if c and os.path.isfile(os.path.join(c, 'scripts', 'run_stage21_colab.py')):
        PROJECT_DIR = c
        break
else:
    hits = glob.glob('/content/drive/MyDrive/**/scripts/run_stage21_colab.py', recursive=True)
    assert hits, 'Project not found. Set PROJECT_DIR manually in this cell.'
    PROJECT_DIR = os.path.dirname(os.path.dirname(hits[0]))

os.chdir(PROJECT_DIR)
print('PROJECT_DIR =', PROJECT_DIR)
print('CPUs:', os.cpu_count())
print(subprocess.run(['nproc'], capture_output=True, text=True).stdout.strip(), 'nproc')

In [ ]:
!pip -q install onnxruntime
!python -c "import numpy, scipy, torch, onnxruntime; print('numpy', numpy.__version__, '| torch', torch.__version__, '| ort', onnxruntime.__version__)"

In [ ]:
# Show what will be resumed
import pickle, os
ckpt = 'results/checkpoints/inverse_records.pkl'
if os.path.exists(ckpt):
    d = pickle.load(open(ckpt, 'rb'))
    print('resuming with', len(d['records']), 'of 2700 realizations already complete')
else:
    print('no checkpoint yet: starting from realization 1')

In [ ]:
# Run (or resume) the inverse study. One worker per vCPU.
!STAGE21_WORKERS=$(nproc) python scripts/run_stage21_colab.py 2>&1 | tee results/stage21_colab_run.log

In [ ]:
# 3-way bias diagnostic (D1 / D2 / D3), using the completed inverse records
!python scripts/run_stage21_bias_real.py 2>&1 | tee results/stage21_bias_run.log

In [ ]:
# Recompute every summary from the individual records
!python scripts/summarize_stage21_real.py 2>&1 | tee results/stage21_summary_run.log

In [ ]:
# Sanity check: diagnostics must be genuinely populated
import json, numpy as np
s = json.load(open('results/stage21_inverse_summary_real.json'))
print('records:', s['n_records'])
for k, v in s['validation'].items():
    print(f"{k:16s} positive/finite {v['n_finite_positive']}/{v['n_total']}  median={v['median']:.3g}")
print('headline (dual @1%):', json.dumps(s.get('overall_1pct', {}), indent=2))

In [ ]:
# Package outputs and download
import shutil
files = [
    'results/stage21_inverse_records_real.npz',
    'results/stage21_inverse_summary_real.json',
    'results/stage21_inverse_metadata.json',
    'results/stage21_coverage_stats.json',
    'results/stage21_bias_diagnostic_real.json',
    'results/table_s22_partB.md',
]
os.makedirs('results/colab_bundle', exist_ok=True)
for f in files:
    if os.path.exists(f):
        shutil.copy2(f, 'results/colab_bundle/')
shutil.make_archive('results/stage21_colab_outputs', 'zip', 'results/colab_bundle')
print(os.listdir('results/colab_bundle'))

from google.colab import files as colab_files
colab_files.download('results/stage21_colab_outputs.zip')